In [1]:
!git clone https://github.com/kuroneko20/banking_ai_agent.git
!ls banking_ai_agent/

Cloning into 'banking_ai_agent'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 32 (delta 0), reused 32 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 27.82 KiB | 791.00 KiB/s, done.
app  examples  README.md  requirements.txt  run.py


In [2]:
!pip install -r banking_ai_agent/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.9/434.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.46.0
    Uninstalling uvicorn-0.46.0:
      Successfully uninstalled uvicorn-0.46.0
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.2
    Uninstalling python-dotenv-1.2.2:
      Successfully uninstalled python-dotenv-1.2.2
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling 

In [3]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,567 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current

In [4]:
import subprocess, threading, time

# 1. Ollama
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Ollama started")

# 2. FastAPI
server = subprocess.Popen(
    ["python", "run.py"],
    cwd="/content/banking_ai_agent",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)
time.sleep(4)

for _ in range(10):
    line = server.stdout.readline().decode(errors="ignore").strip()
    if line: print(line)

✅ Ollama started
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_used" in DraftResult has conflict with protected namespace "model_".
You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
warnings.warn(
INFO:     Started server process [10632]
INFO:     Waiting for application startup.
2026-05-11 16:55:43 | INFO     | app.main | Banking AI-Agent starting up...
2026-05-11 16:55:43 | INFO     | httpx | HTTP Request: GET http://localhost:11434/api/tags "HTTP/1.1 200 OK"
2026-05-11 16:55:43 | INFO     | app.main | ✅ Ollama is reachable at http://localhost:11434 (model: gpt-oss:20b)
INFO:     Application startup complete.


In [5]:
output_lines = []
def read_output(proc):
    for line in proc.stdout:
        output_lines.append(line.decode(errors="ignore").strip())

tunnel = subprocess.Popen([
    "ssh", "-p", "443", "-R", "0:localhost:8000",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "a.pinggy.io"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

threading.Thread(target=read_output, args=(tunnel,), daemon=True).start()
time.sleep(6)

for line in output_lines:
    print(line)

Pseudo-terminal will not be allocated because stdin is not a terminal.
Allocated port 9 for remote forward to localhost:8000
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
http://mihpz-34-125-200-54.run.pinggy-free.link
https://mihpz-34-125-200-54.run.pinggy-free.link


In [6]:
import urllib.request, json

BASE_URL = "https://mihpz-34-125-200-54.run.pinggy-free.link"  # thay URL của bạn

def test_chat(message):
    req = urllib.request.Request(
        f"{BASE_URL}/agent/chat",
        data=json.dumps({"message": message}).encode(),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())

# ── Demo scenarios ──────────────────────────────────────
scenarios = [
    ("🔴 FRAUD",    "I was hacked! There are unauthorized charges on my account!"),
    ("🔴 BLOCKED",  "My account was blocked after a suspicious transfer"),
    ("🔴 LOST CARD","I lost my debit card at the mall, please block it immediately"),
    ("🟡 TRANSFER", "I sent $500 three days ago but receiver still hasn't received it"),
    ("🟡 REFUND",   "I returned a product 2 weeks ago but refund hasn't arrived"),
    ("🟢 BALANCE",  "What is my current account balance?"),
    ("🟢 PASSWORD", "I forgot my password and the reset link is not working"),
    ("🟢 LOAN",     "I want to apply for a personal loan of $10,000"),
]

print("╔══════════════════════════════════════════════════════════════╗")
print("║           🏦  BANKING AI-AGENT — DEMO                       ║")
print("╚══════════════════════════════════════════════════════════════╝")

for label, msg in scenarios:
    res = test_chat(msg)
    intent    = res['intent_result']['intent']
    confidence= res['intent_result']['confidence']
    priority  = res['priority_result']['priority'].upper()
    routing   = res['routing_decision']
    response  = res['final_response']
    latency   = res['total_latency_ms']
    trace     = len(res['workflow_trace'])

    print(f"\n{label}")
    print(f"  📨 Message  : {msg[:55]}")
    print(f"  🎯 Intent   : {intent} ({confidence:.0%} confidence)")
    print(f"  ⚡ Priority  : {priority}")
    print(f"  🔀 Routing  : {routing}")
    print(f"  💬 Response : {response[:80]}...")
    print(f"  ⏱️  Latency  : {latency}ms | 📊 Steps: {trace}")
    print("  " + "─" * 60)

╔══════════════════════════════════════════════════════════════╗
║           🏦  BANKING AI-AGENT — DEMO                       ║
╚══════════════════════════════════════════════════════════════╝

🔴 FRAUD
  📨 Message  : I was hacked! There are unauthorized charges on my acco
  🎯 Intent   : suspicious_transaction (100% confidence)
  ⚡ Priority  : HIGH
  🔀 Routing  : escalate_to_human
  💬 Response : Thank you for contacting us. Given the nature of your request, we are connecting...
  ⏱️  Latency  : 45.08ms | 📊 Steps: 6
  ────────────────────────────────────────────────────────────

🔴 BLOCKED
  📨 Message  : My account was blocked after a suspicious transfer
  🎯 Intent   : blocked_account (100% confidence)
  ⚡ Priority  : HIGH
  🔀 Routing  : escalate_to_human
  💬 Response : Thank you for contacting us. Given the nature of your request, we are connecting...
  ⏱️  Latency  : 44.81ms | 📊 Steps: 6
  ────────────────────────────────────────────────────────────

🔴 LOST CARD
  📨 Message  : I lost my